# Support Ticket Router – train, evaluate and demo on Colab

1. **Runtime → Change runtime type → T4 GPU**
2. Put your GitHub repo URL below and run all cells (≈ 20–30 min).

Training runs on the GPU; all latencies are measured on the CPU (the app is a CPU app).

In [ ]:
REPO_URL = "https://github.com/bsyilmaz/support-ticket-router.git"  # <-- change
!git clone -q $REPO_URL repo
%cd repo
!pip install -q -r requirements.txt

In [ ]:
!nvidia-smi -L
!python train.py 2>&1 | tee train_log.txt

In [ ]:
import json, pandas as pd
R = json.load(open('results.json'))
print(R['hardware'])
rows = [{'model': m['name'], 'in-scope acc': round(100*m['in_scope_accuracy'],2), 'OOS recall': round(100*m['oos_recall'],2),
         'routing acc': round(100*m['routing_accuracy'],2), 'size MB': round(m['size_mb'],1),
         'latency ms (mean)': round(m['latency']['mean_ms'],2), 'tau': m['threshold']} for m in R['models'].values()]
pd.DataFrame(rows)

## Slides
Edit `presentation_info.json` first (left sidebar → Files → double-click it), then run:

In [ ]:
!apt-get -qq install -y libreoffice-impress > /dev/null 2>&1
!python make_diagram.py && python make_slides.py
from google.colab import files
files.download('docs/presentation.pdf'); files.download('docs/architecture.png'); files.download('results.json')

## Publish the weights to the Hugging Face Hub (requirement R7: no weights in git)
Create a *write* token at https://huggingface.co/settings/tokens.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
HF_REPO = "selimyilm/support-ticket-router"  # <-- change
!python upload_artifacts.py $HF_REPO

## Run the app (public link for recording the demo video)
Open the `https://….gradio.live` link printed below. Stop the cell to end the app.

In [ ]:
!GRADIO_SHARE=1 python app.py